# COT_lab — Results & Analysis

Comprehensive results for the CoT knowledge-distillation experiment on **GSM8K** and **SVAMP**,
reproducing and extending Ho et al. (ACL 2023).

## Contents
1. [Setup](#setup)
2. [Training set overview](#training-sets)
3. [GSM8K accuracy — filter ablation](#gsm8k-accuracy)
4. [GSM8K accuracy — model-scale ablation](#scale-ablation)
5. [ReCEval reasoning-quality scores](#receval)
6. [Accuracy × ReCEval correlation](#scatter)
7. [Run-card summary](#runcards)
8. [SVAMP ablation](#svamp)
9. [Key findings & discussion](#discussion)

## Experimental design

| Axis | Values |
|---|---|
| **Dataset** | GSM8K (1 319 test), SVAMP (300 test) |
| **Teacher** | text-davinci-002, zero-shot-CoT (Ho et al. 2023) |
| **Student base** | FLAN-T5-base (250M) |
| **Student large** | FLAN-T5-large (770M) |
| **Training sets** | Set A (unfiltered), Set B (Magister filter), Set C (calc-patched), Direct FT, Set C+Mix |
| **Eval: accuracy** | Final-answer exact match (tolerance 1e-6) |
| **Eval: reasoning** | ReCEval — intra / inter / info (pythia-1b scorer) |

**Reference (Ho et al. 2023, FLAN-T5-base):** baseline 2.50 %, direct FT 5.08 %, CoT FT 4.40 %.


In [ ]:
# <a id="setup"></a>
from pathlib import Path
import json, math, csv

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import numpy as np

REPO_ROOT = Path('..').resolve()
EVAL      = REPO_ROOT / 'outputs' / 'eval_results'
RUNS      = REPO_ROOT / 'outputs' / 'runs'
PLOTS     = REPO_ROOT / 'outputs' / 'plots'
PLOTS.mkdir(parents=True, exist_ok=True)

BASE_ORDER  = ['baseline','student_direct_ft','student_set_a','student_set_b',
               'student_set_c','student_set_c_mix']
LARGE_ORDER = ['student_direct_ft_large','student_set_a_large',
               'student_set_b_large','student_set_c_large']
ALL_ORDER   = BASE_ORDER + LARGE_ORDER

SHORT = {
    'baseline':               'baseline',
    'student_direct_ft':      'direct_ft',
    'student_set_a':          'set_a',
    'student_set_b':          'set_b',
    'student_set_c':          'set_c',
    'student_set_c_mix':      'set_c_mix',
    'student_direct_ft_large':'direct_ft (L)',
    'student_set_a_large':    'set_a (L)',
    'student_set_b_large':    'set_b (L)',
    'student_set_c_large':    'set_c (L)',
}

COLORS = {
    'baseline':               '#aec7e8',
    'student_direct_ft':      '#1f77b4',
    'student_set_a':          '#ffbb78',
    'student_set_b':          '#ff7f0e',
    'student_set_c':          '#2ca02c',
    'student_set_c_mix':      '#98df8a',
    'student_direct_ft_large':'#9467bd',
    'student_set_a_large':    '#c5b0d5',
    'student_set_b_large':    '#d62728',
    'student_set_c_large':    '#8c564b',
}

def _sort(df, order=ALL_ORDER):
    df = df.copy()
    df['_o'] = df['condition'].map({c: i for i, c in enumerate(order)}).fillna(99)
    return df.sort_values('_o').drop(columns='_o').reset_index(drop=True)

def _color_list(conditions):
    return [COLORS.get(c, '#999999') for c in conditions]

acc = _sort(pd.read_csv(EVAL / 'accuracy.csv'))
acc['accuracy_pct'] = (acc['accuracy'] * 100).round(2)
acc['label'] = acc['condition'].map(SHORT)

rec = _sort(pd.read_csv(EVAL / 'receval_summary.csv'))
rec['label'] = rec['condition'].map(SHORT)

print('GSM8K accuracy loaded:  ', len(acc), 'conditions')
print('ReCEval summary loaded: ', len(rec), 'conditions')
print()
print(acc[['condition','correct','accuracy_pct']].to_string(index=False))


## 2. Training Set Overview <a id="training-sets"></a>

All training sets derive from the same Ho et al. teacher (text-davinci-002, zero-shot-CoT) applied to GSM8K train.


In [ ]:
filter_card = json.loads((RUNS / '02_filter.json').read_text())
fm = filter_card['metrics']

train_sets = pd.DataFrame([
    {'Set': 'Direct FT',  'Size (GSM8K)': fm['direct_ft_size'],
     'Filter': 'None — answer-only targets (#### N)',          'CoT': 'No'},
    {'Set': 'Set A',      'Size (GSM8K)': fm['set_a_size'],
     'Filter': 'None — all teacher CoTs kept',                 'CoT': 'Yes'},
    {'Set': 'Set B',      'Size (GSM8K)': fm['set_b_size'],
     'Filter': f'Magister: correct-answer CoTs only ({fm["set_b_keep_rate"]:.1%} kept)', 'CoT': 'Yes'},
    {'Set': 'Set C',      'Size (GSM8K)': fm['set_c_size'],
     'Filter': f'Set B + calculator patches ({fm["n_calculator_edited"]:,} edits = {fm["n_calculator_edited"]/fm["set_b_size"]:.1%})', 'CoT': 'Yes'},
    {'Set': 'Set C+Mix',  'Size (GSM8K)': 6100,
     'Filter': 'Set C CoTs + Direct FT targets blended',       'CoT': 'Mixed'},
])
display(train_sets)

fig, ax = plt.subplots(figsize=(9, 3.2))
colors_ts = ['#1f77b4','#ffbb78','#ff7f0e','#2ca02c','#98df8a']
bars = ax.barh(train_sets['Set'], train_sets['Size (GSM8K)'], color=colors_ts)
for bar, v in zip(bars, train_sets['Size (GSM8K)']):
    ax.text(bar.get_width() + 60, bar.get_y() + bar.get_height()/2,
            f'{v:,}', va='center', fontsize=9)
ax.set_xlabel('Training examples')
ax.set_title('GSM8K training set sizes')
ax.set_xlim(0, 9500)
ax.grid(axis='x', alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS / 'training_set_sizes.png', dpi=150)
plt.show()
print(f"Set B keep rate: {fm['set_b_keep_rate']:.1%}   |   "
      f"Calculator edits on Set C: {fm['n_calculator_edited']:,} / {fm['set_b_size']:,} = "
      f"{fm['n_calculator_edited']/fm['set_b_size']:.1%} of chains")


## 3. GSM8K Accuracy — Filter Ablation <a id="gsm8k-accuracy"></a>

**Question:** Which training set produces the best student on GSM8K at FLAN-T5-base scale?

Reference lines from Ho et al. (2023) on FLAN-T5-base: baseline 2.50 %, direct FT 5.08 %, CoT FT 4.40 %.


In [ ]:
base_acc = _sort(acc[acc['condition'].isin(BASE_ORDER)], order=BASE_ORDER).copy()
display(base_acc[['condition','n','correct','accuracy_pct']].rename(
    columns={'accuracy_pct':'accuracy_%'}))

fig, ax = plt.subplots(figsize=(10, 4))
colors_b = _color_list(base_acc['condition'])
bars = ax.bar(base_acc['label'], base_acc['accuracy_pct'], color=colors_b, width=0.6)

for bar, val, cor in zip(bars, base_acc['accuracy_pct'], base_acc['correct']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.07,
            f'{val:.2f}%\n({cor}/1319)', ha='center', va='bottom', fontsize=7.5)

ax.axhline(4.32, color='#aec7e8', linestyle='--', linewidth=1.2,
           label='Our baseline (4.32%)')
ax.axhline(5.08, color='grey',    linestyle='--', linewidth=1.0,
           label='Ho et al. direct FT (5.08%)')
ax.axhline(4.40, color='grey',    linestyle='-.',  linewidth=1.0,
           label='Ho et al. CoT FT (4.40%)')
ax.axhline(2.50, color='grey',    linestyle=':',   linewidth=0.9,
           label='Ho et al. baseline (2.50%)')

ax.set_ylabel('Accuracy (%)')
ax.set_title('GSM8K accuracy — filter ablation  (FLAN-T5-base, n=1 319)')
ax.set_ylim(0, max(base_acc['accuracy_pct']) * 1.38)
ax.grid(axis='y', alpha=0.3)
ax.legend(fontsize=7.5, loc='upper right')
fig.tight_layout()
fig.savefig(PLOTS / 'gsm8k_filter_ablation.png', dpi=150)
plt.show()


## 4. GSM8K Accuracy — Model-Scale Ablation <a id="scale-ablation"></a>

**Question:** Does FLAN-T5-large (770M) change the ranking? Does CoT catch up to direct FT at larger scale?

Paired bars show the **same training set** at base (250M) vs large (770M).


In [ ]:
pairs = [
    ('direct_ft', 'student_direct_ft',       'student_direct_ft_large'),
    ('set_a',     'student_set_a',            'student_set_a_large'),
    ('set_b',     'student_set_b',            'student_set_b_large'),
    ('set_c',     'student_set_c',            'student_set_c_large'),
]
acc_lookup = acc.set_index('condition')['accuracy_pct'].to_dict()

scale_rows = []
for label, base_c, large_c in pairs:
    b = acc_lookup.get(base_c,  float('nan'))
    l = acc_lookup.get(large_c, float('nan'))
    scale_rows.append({'condition': label, 'base_%': b, 'large_%': l,
                       'delta_pp': round(l - b, 2)})
scale_df = pd.DataFrame(scale_rows)
display(scale_df.round(2))

x = range(len(scale_df))
w = 0.35
fig, ax = plt.subplots(figsize=(9, 4.5))
b1 = ax.bar([i - w/2 for i in x], scale_df['base_%'],  w,
            label='FLAN-T5-base  (250M)', color='steelblue')
b2 = ax.bar([i + w/2 for i in x], scale_df['large_%'], w,
            label='FLAN-T5-large (770M)', color='darkorange')

for bar, val in zip(list(b1)+list(b2),
                    list(scale_df['base_%'])+list(scale_df['large_%'])):
    if not math.isnan(val):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.08,
                f'{val:.2f}%', ha='center', va='bottom', fontsize=7.5)

ax.axhline(4.32, color='#aec7e8', linestyle='--', linewidth=1.0,
           label='Our baseline (4.32%)')
ax.set_xticks(list(x))
ax.set_xticklabels(scale_df['condition'])
ax.set_ylabel('GSM8K accuracy (%)')
ax.set_title('Scale ablation: FLAN-T5-base vs FLAN-T5-large  (n=1 319)')
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(scale_df[['base_%','large_%']].max()) * 1.3)
fig.tight_layout()
fig.savefig(PLOTS / 'gsm8k_scale_ablation.png', dpi=150)
plt.show()

dft_large = acc_lookup.get('student_direct_ft_large', 999)
print("\nDelta (large - base):")
for _, row in scale_df.iterrows():
    beats = ' ← beats direct_ft_large!' if row['condition'] != 'direct_ft' and row['large_%'] > dft_large else ''
    print(f"  {row['condition']:12s}  +{row['delta_pp']:.2f} pp{beats}")


## 5. ReCEval — Reasoning Chain Quality <a id="receval"></a>

Three chain-level metrics on all 1 319 GSM8K test generations:

| Metric | Measures | Interpretation |
|---|---|---|
| **intra** | NLI entailment between consecutive steps | Step-level self-consistency (higher = more coherent) |
| **inter** | Max contradiction between a step and all prior steps | Inter-step coherence (higher = less contradictory) |
| **info** | Log-perplexity delta (pythia-1b) when each step is added | How much steps help predict gold answer (positive = informative) |

Info scorer: **EleutherAI/pythia-1b**.


In [ ]:
display(rec[['condition','n','intra_mean','intra_std','inter_mean','inter_std',
             'info_mean','info_std']].round(4))


In [ ]:
METRIC_META = [
    ('intra', 'Intra-step coherence',       (0.88, 1.0)),
    ('inter', 'Inter-step relevance',        None),
    ('info',  'Informativeness (pythia-1b)', None),
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (m, title, ylim) in zip(axes, METRIC_META):
    means = rec[f'{m}_mean'].tolist()
    stds  = [v if not (isinstance(v, float) and math.isnan(v)) else 0
             for v in rec[f'{m}_std'].tolist()]
    ax.bar(range(len(rec)), means, yerr=stds, color=_color_list(rec['condition']),
           capsize=3, error_kw={'linewidth': 0.8})
    ax.set_xticks(range(len(rec)))
    ax.set_xticklabels(rec['label'], rotation=35, ha='right', fontsize=7.5)
    ax.set_title(title, fontsize=10)
    if ylim:
        ax.set_ylim(*ylim)
    if m == 'info':
        ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.grid(axis='y', alpha=0.3)

handles = [mpatches.Patch(color=COLORS[c], label=SHORT[c])
           for c in ALL_ORDER if c in set(rec['condition'])]
fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=7.5,
           bbox_to_anchor=(0.5, -0.05))
fig.suptitle('ReCEval means ± 1 std  (GSM8K, n=1 319)', fontsize=12)
fig.tight_layout(rect=[0, 0.07, 1, 1])
fig.savefig(PLOTS / 'receval_bars.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Base vs large ReCEval delta table
receval_pairs = [('direct_ft','student_direct_ft','student_direct_ft_large'),
                 ('set_a',    'student_set_a',    'student_set_a_large'),
                 ('set_b',    'student_set_b',    'student_set_b_large'),
                 ('set_c',    'student_set_c',    'student_set_c_large')]

rec_lookup = rec.set_index('condition')
rows_rc = []
for label, bc, lc in receval_pairs:
    for m in ['intra','inter','info']:
        bv = float(rec_lookup.loc[bc, f'{m}_mean']) if bc in rec_lookup.index else float('nan')
        lv = float(rec_lookup.loc[lc, f'{m}_mean']) if lc in rec_lookup.index else float('nan')
        rows_rc.append({'condition': label, 'metric': m,
                        'base': round(bv,4), 'large': round(lv,4),
                        'delta': round(lv - bv, 4)})
rc_df = pd.DataFrame(rows_rc)
display(rc_df.pivot(index='condition', columns='metric', values=['base','large','delta']).round(4))


## 6. Accuracy × ReCEval Correlation <a id="scatter"></a>

Does higher GSM8K accuracy correlate with better reasoning-quality scores?
Each point is one condition.


In [ ]:
merged = acc[acc['condition'].isin(rec['condition'])][['condition','accuracy_pct','label']].merge(
    rec[['condition','intra_mean','inter_mean','info_mean']], on='condition'
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (m, title, _) in zip(axes, METRIC_META):
    for _, row in merged.iterrows():
        ax.scatter(row['accuracy_pct'], row[f'{m}_mean'],
                   c=COLORS.get(row['condition'],'grey'), s=80, zorder=3,
                   edgecolors='white', linewidths=0.5)
        ax.annotate(row['label'], (row['accuracy_pct'], row[f'{m}_mean']),
                    fontsize=6.5, xytext=(4, 2), textcoords='offset points')
    if m == 'info':
        ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
    ax.set_xlabel('GSM8K accuracy (%)')
    ax.set_ylabel(m)
    ax.set_title(f'Accuracy vs {title}', fontsize=9)
    ax.grid(alpha=0.25)

handles = [mpatches.Patch(color=COLORS[c], label=SHORT[c])
           for c in ALL_ORDER if c in set(merged['condition'])]
fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=7,
           bbox_to_anchor=(0.5, -0.07))
fig.tight_layout(rect=[0, 0.08, 1, 1])
fig.savefig(PLOTS / 'accuracy_vs_receval.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. Run-card Summary <a id="runcards"></a>

Training metadata from `outputs/runs/`. Duration in minutes.


In [ ]:
rows_rc2 = []
for cond in ALL_ORDER:
    row = {'condition': cond, 'label': SHORT[cond]}
    for stage, prefix in [('train','03_train'),('infer','04_inference'),('receval','05b')]:
        candidates = [RUNS / f'{prefix}_{cond}.json',
                      RUNS / f'{prefix}_gsm8k_{cond}.json']
        cp = next((c for c in candidates if c.exists()), None)
        if cp:
            d = json.loads(cp.read_text())
            row[f'{stage}_status'] = d.get('status','?')
            row[f'{stage}_min'] = round(d.get('duration_seconds', float('nan')) / 60, 1)
            if stage == 'train':
                m = d.get('metrics', {})
                row['n_train']   = m.get('n_train', '')
                row['best_loss'] = round(m.get('best_eval_loss', float('nan')), 4)
                row['epochs']    = round(m.get('n_epochs_completed', float('nan')), 1)
        else:
            row[f'{stage}_status'] = '—'
            row[f'{stage}_min'] = float('nan')
    rows_rc2.append(row)

df_rc = pd.DataFrame(rows_rc2)
display(df_rc[['label','n_train','epochs','best_loss',
               'train_min','infer_min','receval_min',
               'train_status','infer_status','receval_status']])


## 8. SVAMP Ablation <a id="svamp"></a>

**SVAMP** (300-example test set): simpler 1–2-step arithmetic word problems.
Same teacher (text-davinci-002, zero-shot-CoT); teacher accuracy ~79 % vs ~82 % on GSM8K.

### Training data (SVAMP)

| Set | Size | Notes |
|---|---|---|
| Direct FT | 700 | All SVAMP train questions, answer-only |
| Set A | 509 | Teacher CoTs with matching question (191 had no teacher record) |
| Set B | 299 | Magister filter: 58.7 % of Set A kept |
| Set C | 299 | Set B + calculator patches (19 edits — near-identical to Set B) |

### Two experimental axes

| Axis | Conditions |
|---|---|
| **In-distribution** | Train on SVAMP → test on SVAMP (`svamp_student_*`) |
| **Cross-transfer** | GSM8K-trained → SVAMP test, no retraining (`baseline`, `student_set_b`) |

Run `notebooks/Kaggle_svamp.ipynb` on Kaggle to generate these results.


In [ ]:
SVAMP_EVAL = EVAL / 'svamp'
svamp_csv  = SVAMP_EVAL / 'svamp_accuracy.csv'

SVAMP_ORDER = ['baseline','student_set_b',
               'svamp_student_direct_ft','svamp_student_set_a',
               'svamp_student_set_b','svamp_student_set_c']
SVAMP_SHORT = {
    'baseline':               'baseline\n(zero-shot)',
    'student_set_b':          'set_b\n(GSM8K->SVAMP)',
    'svamp_student_direct_ft':'direct_ft\n(SVAMP)',
    'svamp_student_set_a':    'set_a\n(SVAMP)',
    'svamp_student_set_b':    'set_b\n(SVAMP)',
    'svamp_student_set_c':    'set_c\n(SVAMP)',
}
SVAMP_COLORS = {
    'baseline':               '#aec7e8',
    'student_set_b':          '#6baed6',
    'svamp_student_direct_ft':'#1f77b4',
    'svamp_student_set_a':    '#ffbb78',
    'svamp_student_set_b':    '#ff7f0e',
    'svamp_student_set_c':    '#2ca02c',
}

if not svamp_csv.exists():
    print('SVAMP results not found — run Kaggle_svamp.ipynb first.')
else:
    def _sort_svamp(df):
        df = df.copy()
        df['_o'] = df['condition'].map({c:i for i,c in enumerate(SVAMP_ORDER)}).fillna(99)
        return df.sort_values('_o').drop(columns='_o').reset_index(drop=True)

    svamp_acc = _sort_svamp(pd.read_csv(svamp_csv))
    svamp_acc['accuracy_pct'] = (svamp_acc['accuracy'] * 100).round(2)
    svamp_acc['label'] = svamp_acc['condition'].map(SVAMP_SHORT)
    display(svamp_acc[['condition','n','correct','accuracy_pct']])


In [ ]:
if svamp_csv.exists():
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

    # === Left: SVAMP test accuracy, coloured by training source ===
    # Three categories: no training, GSM8K-trained (cross-transfer), SVAMP-trained (in-dist).
    CATEGORY_COLOR = {
        'baseline':                'lightgrey',   # no training
        'student_set_b':           'steelblue',   # GSM8K-trained, evaluated on SVAMP
        'svamp_student_direct_ft': 'tomato',      # SVAMP-trained (in-distribution)
        'svamp_student_set_a':     'tomato',
        'svamp_student_set_b':     'tomato',
        'svamp_student_set_c':     'tomato',
    }
    ax = axes[0]
    bar_colors = [CATEGORY_COLOR.get(c, '#999') for c in svamp_acc['condition']]
    bars = ax.bar(range(len(svamp_acc)), svamp_acc['accuracy_pct'],
                  color=bar_colors, width=0.6,
                  edgecolor='dimgrey', linewidth=0.5)
    ax.set_xticks(range(len(svamp_acc)))
    ax.set_xticklabels(svamp_acc['label'], fontsize=8)
    ax.set_ylabel('SVAMP test accuracy (%)')
    ax.set_title('SVAMP test accuracy by training source  (n=300)')
    ax.grid(axis='y', alpha=0.3)

    bl = float(svamp_acc[svamp_acc['condition'] == 'baseline']['accuracy_pct'].iloc[0])
    ax.axhline(bl, color='dimgrey', linestyle=':', linewidth=0.8)

    for bar, val, cor in zip(bars, svamp_acc['accuracy_pct'], svamp_acc['correct']):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                f'{val:.1f}%\n({cor})', ha='center', va='bottom', fontsize=7.5)

    ax.legend(handles=[
        mpatches.Patch(color='lightgrey', label='Zero-shot baseline (no training)'),
        mpatches.Patch(color='steelblue', label='Cross-transfer (GSM8K-trained, eval on SVAMP)'),
        mpatches.Patch(color='tomato',    label='In-distribution (SVAMP-trained)'),
    ], fontsize=7.5, loc='upper left')

    # === Right: paired GSM8K vs SVAMP for the conditions evaluated on BOTH ===
    # Only `baseline` and GSM8K-trained `student_set_b` were evaluated on the SVAMP test.
    # The other GSM8K student conditions (direct_ft / set_a / set_c) were never
    # cross-evaluated, so they would be empty bars and are intentionally omitted.
    cross_conditions = ['baseline', 'student_set_b']
    cross_labels     = ['baseline\n(zero-shot)', 'set_b\n(GSM8K-trained)']
    gsm_vals   = [float(acc.set_index('condition')['accuracy_pct'].get(c, float('nan')))
                  for c in cross_conditions]
    svamp_vals = [float(svamp_acc.set_index('condition')['accuracy_pct'].get(c, float('nan')))
                  for c in cross_conditions]

    ax2 = axes[1]
    x2 = list(range(len(cross_conditions)))
    w2 = 0.35
    b1 = ax2.bar([i - w2 / 2 for i in x2], gsm_vals,  w2,
                 label='GSM8K test (n=1 319)', color='steelblue')
    b2 = ax2.bar([i + w2 / 2 for i in x2], svamp_vals, w2,
                 label='SVAMP test (n=300)',   color='tomato', alpha=0.85)
    for bars_pair, vals in [(b1, gsm_vals), (b2, svamp_vals)]:
        for bar, v in zip(bars_pair, vals):
            ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                     f'{v:.2f}%', ha='center', va='bottom', fontsize=8)
    ax2.set_xticks(x2)
    ax2.set_xticklabels(cross_labels, fontsize=9)
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Cross-dataset transfer:\nsame model evaluated on GSM8K vs SVAMP')
    ax2.legend(fontsize=8, loc='upper right')
    ax2.grid(axis='y', alpha=0.3)

    fig.tight_layout()
    fig.savefig(PLOTS / 'svamp_overview.png', dpi=150)
    plt.show()


In [ ]:
if svamp_csv.exists():
    svamp_lk = svamp_acc.set_index('condition')['accuracy_pct'].to_dict()
    baseline_acc = float(svamp_lk['baseline'])

    # --- Question A: how much does SVAMP-training help vs no training? ---
    in_dist_rows = [
        ('direct_ft', 'svamp_student_direct_ft'),
        ('set_a',     'svamp_student_set_a'),
        ('set_b',     'svamp_student_set_b'),
        ('set_c',     'svamp_student_set_c'),
    ]
    in_dist_df = pd.DataFrame([{
        'condition': label,
        'zero_shot_baseline_%': round(baseline_acc, 2),
        'svamp_trained_%': round(svamp_lk.get(sc, float('nan')), 2),
        'training_gain_pp': round(svamp_lk.get(sc, float('nan')) - baseline_acc, 2),
    } for label, sc in in_dist_rows])
    print("A. In-distribution SVAMP training (vs zero-shot baseline):")
    display(in_dist_df)

    # --- Question B: does a GSM8K-trained model transfer to SVAMP at all? ---
    cross_rows = [('set_b', 'student_set_b')]
    cross_df = pd.DataFrame([{
        'condition': label,
        'zero_shot_baseline_%': round(baseline_acc, 2),
        'gsm8k_trained_on_svamp_%': round(svamp_lk.get(cc, float('nan')), 2),
        'transfer_delta_pp': round(svamp_lk.get(cc, float('nan')) - baseline_acc, 2),
    } for label, cc in cross_rows])
    print("")
    print("B. Cross-dataset transfer (GSM8K-trained, evaluated on SVAMP, no retraining):")
    display(cross_df)

    # --- Plot: paired baseline-vs-in-dist bars, plus a star marking the
    # actual cross-transfer point for the one condition we tested (set_b). ---
    x3 = list(range(len(in_dist_df)))
    fig, ax = plt.subplots(figsize=(8.5, 4.5))

    ax.bar([i - 0.2 for i in x3], in_dist_df['zero_shot_baseline_%'], 0.35,
           label='Zero-shot baseline (no training)', color='lightgrey',
           edgecolor='dimgrey', linewidth=0.6)
    ax.bar([i + 0.2 for i in x3], in_dist_df['svamp_trained_%'], 0.35,
           label='SVAMP-trained (in-distribution)', color='tomato', alpha=0.85)

    set_b_idx = list(in_dist_df['condition']).index('set_b')
    cross_val = float(svamp_lk['student_set_b'])
    ax.scatter([set_b_idx - 0.2], [cross_val],
               marker='*', s=260, color='steelblue', zorder=5,
               edgecolors='black', linewidths=0.6,
               label=f'GSM8K-trained set_b on SVAMP ({cross_val:.2f}%, below baseline = negative transfer)')

    for i, (g, v) in enumerate(zip(in_dist_df['training_gain_pp'],
                                    in_dist_df['svamp_trained_%'])):
        sign = '+' if g >= 0 else ''
        ax.text(i + 0.2, v + 0.15, f'{sign}{g:.1f} pp',
                ha='center', va='bottom', fontsize=8)

    ax.axhline(baseline_acc, color='dimgrey', linestyle=':', linewidth=0.8)
    ax.set_xticks(x3)
    ax.set_xticklabels(in_dist_df['condition'])
    ax.set_ylabel('SVAMP accuracy (%)')
    ax.set_title('SVAMP: training gain from in-distribution fine-tuning\n'
                 '(star = the one GSM8K-trained model evaluated cross-dataset)')
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(axis='y', alpha=0.3)
    fig.tight_layout()
    fig.savefig(PLOTS / 'svamp_transfer_gap.png', dpi=150)
    plt.show()


## 9. Key Findings & Discussion <a id="discussion"></a>

---

### 9.1 Filter ablation (GSM8K, FLAN-T5-base)

| Finding | Evidence |
|---|---|
| **Direct FT is the strongest strategy at base scale** | `direct_ft` 5.23 % vs `baseline` 4.32 % (+0.91 pp). Matches Ho et al. direct FT (5.08 %). |
| **All CoT sets underperform the baseline** | Set A 2.65 %, Set B 3.34 %, Set C 3.03 % — all below 4.32 % baseline. |
| **Magister filter (Set B) recovers ~0.7 pp over Set A** | Filtering wrong-answer CoTs 7 473 → 3 389 raises accuracy from 2.65 % to 3.34 %. Quality > quantity for CoT distillation at small scale. |
| **Calculator patching (Set C) adds no signal** | 676 edits on 3 389 chains (20 %) gives 3.03 % — marginally *worse* than Set B 3.34 %. Teacher arithmetic errors are correlated with hard questions the student cannot solve regardless. |
| **Set C+Mix is not beneficial** | 3.11 % — neither reaches direct FT (5.23 %) nor improves over Set B (3.34 %). |

**Interpretation:** At 250M params, FLAN-T5-base lacks the capacity to leverage multi-step CoT supervision. Reproduces Ho et al.'s scale threshold finding.

---

### 9.2 Scale ablation (GSM8K, FLAN-T5-large vs base)

| Condition | Base (250M) | Large (770M) | Delta |
|---|---|---|---|
| direct_ft  | 5.23 % | **6.90 %** | +1.67 pp |
| set_a      | 2.65 % | **5.99 %** | +3.34 pp |
| set_b      | 3.34 % | **5.16 %** | +1.82 pp |
| set_c      | 3.03 % | **4.70 %** | +1.67 pp |

| Finding | Evidence |
|---|---|
| **Set A large has the biggest scale gain (+3.34 pp)** | From worst base condition (2.65 %) to competitive large condition (5.99 %). The large model can recover from noisy CoT data that breaks the small model. |
| **Set A large (5.99 %) beats Set B large (5.16 %)** | Unfiltered CoT becomes the second-best condition at 770M scale. Magister filtering is less necessary as model capacity increases. |
| **CoT gap narrows but does not close** | `set_b_large` 5.16 % vs `direct_ft_large` 6.90 %: CoT is still −1.74 pp behind at 770M. Ho's crossover was observed at GPT-2 scale (~1.5B+). |
| **Set B large finally beats our baseline** | 5.16 % > 4.32 % — CoT distillation becomes net-positive at T5-large scale for the first time. |
| **Set C still trails Set B at large scale** | 4.70 % vs 5.16 % — calculator correction remains unhelpful at both scales. |

---

### 9.3 ReCEval reasoning quality

| Finding | Evidence |
|---|---|
| **Info score is the only discriminative metric** | Direct FT: +0.24 (base), +0.31 (large). All CoT conditions: ≈ −2.2. Negative = steps hurt gold-answer prediction. |
| **Positive info does not mean good reasoning** | Direct FT generates short "#### N" outputs. pythia-1b predicts these more easily than multi-sentence CoT, inflating the score — this is a scorer artefact. |
| **Set C inter > Set B inter at base scale** | 0.203 vs 0.061. Calculator-patched chains are less contradictory step-to-step, even though they give lower accuracy. |
| **Inter collapses at large scale for Set B/C** | `set_b_large` inter 0.053, `set_c_large` inter 0.051. Large models generate longer chains with more steps, increasing contradiction probability. |
| **Set A large inter is higher than Set B/C large** | 0.071 vs 0.053/0.051. Unfiltered CoT may produce more varied step structures that happen to contradict less at large scale. |
| **Intra is near-ceiling and uninformative** | 0.91–0.97 across all conditions. NLI with (step, step) trivially entails itself. |

---

### 9.4 SVAMP cross-dataset ablation

| Finding | Evidence |
|---|---|
| **In-distribution training gives large gains** | SVAMP direct_ft 7.0 % vs 3.0 % baseline (+133 %). Set B 3.67 %, Set C 4.0 % — both above baseline. |
| **Set A fails even in-distribution** | `svamp_student_set_a` = 3.0 % = baseline. Unfiltered CoT hurts regardless of dataset or task difficulty. |
| **Set C > Set B on SVAMP (unlike GSM8K)** | 4.0 % vs 3.67 %. SVAMP has 1–2 step arithmetic where corrected intermediate steps are directly actionable. |
| **GSM8K-trained Set B negatively transfers** | `student_set_b` → SVAMP: 2.67 % < 3.0 % baseline. Complex GSM8K reasoning patterns actively interfere with simpler SVAMP problems. |

---

### 9.5 Limitations

| Limitation | Impact |
|---|---|
| **Scale threshold not reached** | T5-large (770M) is below Ho's crossover point. Cannot confirm CoT advantage at GPT-2 scale (1.5B+). |
| **Small test sets** | GSM8K n=1 319, SVAMP n=300. A 1-example difference = 0.33 pp on SVAMP, making small gaps statistically fragile. |
| **ReCEval info scorer confound** | pythia-1b is biased toward short outputs. All info score conclusions are qualified by this artefact. |
| **Single seed** | All experiments use seed 42. Variance across seeds is unknown. |
| **Cross-transfer limited** | Only `student_set_b` was cross-evaluated on SVAMP. Full transfer sweep needs all GSM8K checkpoints to be retained. |
